# README
- **Author**: RUEI-HSIANG, CHANG（B12303030；AI協助）
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
第九輪核對10人的政府人物／角色名錄入口。9人以具名鄉鎮職務及行政區碼連到2022其他民選公職名冊，補完整生日、明載性別、學歷級別及非首次任何公職。黃宣寓只有吉安調解委員具名簡歷與明載男，不推定生日／學歷，也不把調解委員當民選公職。整頁文字其他人的學經歷不套入本人。

## What does this file take?
PINS固定3份前輪快照、10人人工審閱設定、政府原始HTML、CEC申報CSV及行政區碼。3份新HTTP嘗試與TPP工具可見段落分開保存；TPP HTTP403不冒稱成功下載。大多政府原始來源是前輪已保存，本輪重新審閱，不宣稱新下載。

## What does this file output?
3份current整合表、10人審閱表與逐欄證據，共5份CSV。角色是review_inventory/evidence，非最終分析樣本。人物／官方名錄入口缺口預計730降至720；其中名錄只有角色，不等於完整學經歷。

## Known limitations
不改登記黨派或2026在任欄；2022其他公職參選不能當作曾參選議員。出生日期照官方申報。學歷為2022申報級別，不認證2026最高學位；TPP學校清單不自動解讀畢業。Meta及貼文窗口暫緩。


公開來源副本修訂（2026-10-19）：7份第三方HTML中的Google地圖API key已遮蔽；原始HTTP回應bytes已核對後私存於Git儲存庫外。來源manifest同時保存original_response_sha256與清理後sha256，PINS及輸出來源指紋綁定清理後公開版本。人物學經歷、生日、政見等研究文字未刪改。詳見[清理紀錄](../../report/source_capture_security_20261019.md)。

In [1]:
from pathlib import Path
import pandas as pd,json,hashlib,csv,datetime,re,unicodedata
from bs4 import BeautifulSoup
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
def read(p):return pd.read_csv(ROOT/p,dtype=str,keep_default_na=False)
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',str(s)))
def rows(p):
 with (ROOT/p).open(encoding='utf-8-sig') as f:return [[s.strip().lstrip("'") for s in r] for r in csv.reader(f)]
PINS={'data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv': '4fba9288b89aa417734ea686b09ccad5eed147b47162265c799319bd025fa926', 'data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv': '80d9ccb7ea2cb1e5adfd3488f0b3d16581b067311d9e230f3daee37cf7b059ad', 'data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv': '980d3f2c01d304ed4c9c94128602fb2f55ae8429516b58e92ddbe13886240831', 'data/code/municipal_roles_review_round9_20261019.json': 'ce45941f035646008ec98a41aaaa337d8c1819752c2d853000e5ef67bf769e56', 'data/processed/council_career_source_worklist_round6_20261019.csv': '7f467321214ecb4c640dd66e951ac16ea9663b4072f075213cebb530263a542e', 'data/raw/cec_all_offices_manifest.json': '9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231', 'data/raw/cec_all_offices/101_elcand.csv': '35853ac9d8a7eee4e9882bc04148462876110dcdb3cf0cdc11af0908e1c800f0', 'data/raw/cec_all_offices/103_elcand.csv': '46773806f1b6ef6389e632be7c10ea59bcb5c3afff3ad2715787974eafe494f0', 'data/raw/cec_all_offices/112_elcand.csv': 'f355705254d5ca457d4fe130b77983b769602126b10713e7c968889d2d52530c', 'data/raw/cross_office_round8_20261019/2022_village_elbase.csv': '4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79', 'data/raw/municipal_roles_round9_20261019/82e3fb55c475911e453d.html': 'ee5eccb74a99bc10078257e62d2a98fc4986eab743c853b7ce5f9debb56ab3e3', 'data/raw/municipal_roles_round9_20261019/6556882ec6ac55681d90.html': '027ddae3c619ea334c71cf921f67e0a119822e547f351f40b894d9a37eb12a9e', 'data/raw/municipal_roles_round9_20261019/d56e6a8a8d5ccaf32087.html': 'e8aa0d5deac638ff4b1be04646df7c2795520e1b2d2c38fe44bca6b5e460be09', 'data/raw/municipal_roles_round9_20261019/2022_R1_elbase.csv': '96ee98216f80e97cee27c8edbbbab35c240717bdd2ecf5eef4edaea4b45d36ee', 'data/raw/municipal_roles_round9_20261019/2022_D1_elbase.csv': '780f22895ffc9db630f92733fce3defbe913e6c936ca64cecf8860e34df0768a', 'data/raw/municipal_roles_round9_20261019/manifest.json': 'b32a0ca59f85bfd184e2f81af88e5c416430cc927196b280f238d31b7c2fb820', 'data/raw/municipal_roles_round9_20261019/extraction_manifest.json': '7f036b1e2f81e98e11cd54e36505f2c67b12ffc29ac7683dba8885a58c113b5f', 'data/raw/municipal_roles_round9_20261019/tpp_zhuang_web_excerpt.json': '02bb213dd07af91d2090223b74aea4164f51d02196faab68f5a3e0fe976c2a4f', 'data/raw/career_expansion_20261019/b587a5e46c6e1f6751ab.source': '546bf7a06c22f241135b6d64454d4d79615684ec3cb5f7679f053b59361311de', 'data/raw/career_expansion_20261019/e3a22c5d3489daf8cda3.source': '6ef3258faceba1559d55f5b3b33bedb3b5c5913ff69785351ffb4e7130fd67fb', 'data/raw/career_expansion_20261019/ae8adaeaad8ae7afaa36.source': '4cf7c9437b56e9757ba167f8f37a7e8b9375beac6b35b0e4566509fe28c322da', 'data/raw/career_expansion_20261019/64937ab1e916ff2df53f.source': 'a82ac876aea775810ba88270c917de751a00500e0b5af08d504ff6fbcc85b67e', 'data/raw/career_expansion_20261019/90edbc3b20226a41f3a8.source': 'ccc5959c2ab818c43776497d2c8e631553a2203097dabd8b1932f06063a6b66e', 'data/raw/career_expansion_20261019/0912e18a119d299531bf.source': 'b9f023b3b2aa30d147b291e3eaaa717987ac5ad5a381bb3ddde304ebaba73784', 'data/raw/career_expansion_20261019/606002eb4a668a9a18ea.source': '6a0a025642a80743ce91058c250cb8e3ae1e724569209f26667bd409624c8108', 'data/raw/career_expansion_20261019/6b8915ef8fb902feabfa.source': '6b20ad535e95fba23c662a8db3c4de71ef6d5677e95b9f2b93bcb02df11f67b9', 'data/raw/career_expansion_20261019/8fb25c97b1da43ae8bd0.source': 'dbd04ba156909aa18bc3e3d8e597a16d8a02a6ec2b13adb66ad1d0695633c019'}
S='data/processed/snapshots/pre_municipal_round9_20261019'
for p,h in PINS.items():assert sha(p)==h,p
master=read(S+'/candidate_master_current.csv');fields=read(S+'/candidate_fields_current.csv');council=read(S+'/council_background_20261019.csv');before=council.copy(deep=True)
plan=json.loads((ROOT/'data/code/municipal_roles_review_round9_20261019.json').read_text());obs=[];review=[]
def setfield(key,f,v,status,url,note):
 m=fields.candidate_key.eq(key)&fields.field.eq(f);assert m.sum()==1
 fields.loc[m,['existing_value','status','evidence','note']]=[v,status,url,note];master.loc[master.candidate_key.eq(key),f+'_status']=status
def ev(a,f,v,p,url,row='',note=''):
 obs.append({'candidate_key':a['candidate_key'],'candidate_name':a['candidate_name'],'field':f,'value':v,'source_file':p,'source_url':url,'source_row':str(row),'evidence_status':note,'checked_date':'2026-10-19'})
print('Pinned input identities verified:',len(PINS))


Pinned input identities verified: 27


In [2]:
for a in plan:
 key=a['candidate_key'];name=a['candidate_name'];m=council.candidate_key.eq(key);assert m.sum()==1;c=council[m].iloc[0]
 assert c.county_city==a['county_city'] and c.electoral_district==a['electoral_district'] and c.candidate_name==name
 assert c.moi_profile_url==c.campaign_profile_url==c.additional_profile_url==''
 soup=BeautifulSoup((ROOT/a['source_file']).read_text(errors='replace'),'html.parser');[x.decompose() for x in soup(['script','style','nav','footer','header'])];body=soup.get_text(' ',strip=True)
 assert norm(a['context_anchor']) in norm(body) and norm(a['person_anchor']) in norm(body),(name,a['person_anchor'])
 # Explicitly classify a person-role roster separately from a full individual biography.
 council.loc[m,['additional_profile_url','additional_profile_status','experience_raw','background_source_url']]=[a['source_url'],'government_person_or_role_entry; roster content is not a full biography; identity based on exact locality and named role',a['experience_reviewed'],a['source_url']]
 setfield(key,'profile',a['source_url'],'政府具名人物／職務名錄入口已有，完整履歷另核對',a['source_url'],a['identity_review'])
 ev(a,'profile',a['source_url'],a['source_file'],a['source_url'],note=a['identity_review'])
 ev(a,'experience',a['experience_reviewed'],a['source_file'],a['source_url'],note='由具名段落審閱；不包含其他人職務，不推定2026任職法律狀態')
 if a['cec_file']:
  hits=[(i+1,r) for i,r in enumerate(rows(a['cec_file'])) if len(r)>12 and r[6]==name and '|'.join(r[:5])==a['cec_geo']];assert len(hits)==1
  line,r=hits[0];roc=r[9].zfill(7);birth=datetime.date(int(roc[:3])+1911,int(roc[3:5]),int(roc[5:])).isoformat();sex={'1':'男','2':'女'}[r[8]]
  assert birth==a['birth_date_expected'] and r[12]==a['education_level_2022'] and r[14]=='*'
  codes=r[:5];gp='data/raw/municipal_roles_round9_20261019/2022_'+('D1' if '/101_' in a['cec_file'] else 'R1')+'_elbase.csv' if '/112_' not in a['cec_file'] else 'data/raw/cross_office_round8_20261019/2022_village_elbase.csv'
  gr=rows(gp);geo_matches=[z for z in gr if z[:5]==codes];assert len(geo_matches)==1
  loc=geo_matches[0][5]
  expectedloc={'吳仁祥':'馬公','莊國輝':'馬公','宋萬富':'白沙','李家萱':'西嶼','蔡志堅':'樹下','曾茂照':'三星','黃士駿':'蘇澳','吳新文':'玉里','李麗美':'湖西'}[name]
  # R1 full election-area row may not include town name: check the parent town row as well.
  geotext=' '.join(z[5] for z in gr if z[:2]==codes[:2] and z[3]==codes[3])
  assert expectedloc in geotext,(name,geotext)
  council.loc[m,['birth_date_additional_official','birth_date_additional_source_url','birth_date_additional_source_year','birth_date_additional_evidence_status','sex_additional_source_value','sex_additional_source_url','sex_additional_source_year','additional_historical_education_cec','education_raw','first_time_any_office_supported']]=[birth,'https://db.cec.gov.tw/','2022','official other-office registration; named government role and exact town/village code identity support',sex,'https://db.cec.gov.tw/','2022',r[12],r[12],'0']
  for f,v in [('birth_date',birth),('sex',sex),('education',r[12]),('first_time_any_office','0')]:
   setfield(key,f,v,'2022其他民選公職申報與政府具名職務／地域支持','https://db.cec.gov.tw/ | '+a['source_url'],a['identity_review']+'；學歷僅當年級別；其他公職不當作曾參選議員')
   ev(a,f,v,a['cec_file'],'https://db.cec.gov.tw/',line,'2022官方歷史申報；'+a['identity_review'])
  if name in ['吳仁祥','莊國輝']:
   # Two primary government sources explicitly agree on their complete birthday.
   assert norm(a['person_anchor']) in norm(body)
  conclusion='2022_other_elected_office_supported_not_first_any; first_council_unknown'
 else:
  assert name=='黃宣寓' and norm('姓名：黃宣寓性別：男') in norm(body)
  birth='';sex='男';line='';loc='吉安鄉調解委員會'
  council.loc[m,['sex_additional_source_value','sex_additional_source_url','sex_additional_source_year']]=['男',a['source_url'],'not_stated']
  setfield(key,'sex','男','政府調解委員名錄個人區塊明載男',a['source_url'],'不由姓名／照片推測；調解委員不是民選身分，不推出非首次參選')
  ev(a,'sex','男',a['source_file'],a['source_url'],note='政府名錄個人區塊明載；來源年份未明列')
  conclusion='sex_and_named_career_only; birth_education_first_candidacy_unknown'
 review.append({**{k:a[k] for k in ['candidate_key','candidate_name','county_city','electoral_district','source_url','source_file','context_anchor','person_anchor','identity_review']},'source_type':'government_named_person_or_role_roster_not_full_bio','cec_file':a['cec_file'],'cec_row':str(line),'cec_locality':loc,'birth_date':birth,'sex':sex,'education_level_2022':a['education_level_2022'],'experience':a['experience_reviewed'],'decision':conclusion})
# Party page names schools without saying completed degree: store as a second source, not a degree award.
x=json.loads((ROOT/'data/raw/municipal_roles_round9_20261019/tpp_zhuang_web_excerpt.json').read_text());assert '澎湖縣｜第一選區（馬公市）' in x['visible_text']
a=next(a for a in plan if a['candidate_name']=='莊國輝');key=a['candidate_key'];edu='國立臺北科技大學經營管理學系 | 國立馬公高級中學資料處理科 | 澎湖縣立文光國中 | 澎湖縣立馬公國小';exp='台灣民眾黨澎湖縣黨部執行長 | 台灣民眾黨澎湖縣第一屆黨代表 | 網路公司行銷策略主管'
for f,v in [('education',edu),('experience',exp)]:ev(a,f,v,'data/raw/municipal_roles_round9_20261019/tpp_zhuang_web_excerpt.json',x['url'],note='當屆政黨候選人頁工具可見摘錄；原始HTTP403分開保留；學校清單未自動認定畢業')
# Supplementary current election role evidence is only used when actually present in the HTTP body.
for cap in json.loads((ROOT/'data/raw/municipal_roles_round9_20261019/manifest.json').read_text()):
 if cap['status']!='200':continue
 text=BeautifulSoup((ROOT/cap['path']).read_text(errors='replace'),'html.parser').get_text(' ',strip=True)
 name=cap['candidate_name'];a=next(a for a in plan if a['candidate_name']==name)
 if name=='蔡志堅':assert '樹下里長蔡志堅' in text and '香山區市議員' in text
 if name=='宋萬富':assert '參選澎湖縣議員' in text and '白沙' in text
 ev(a,'current_identity_context',name,cap['path'],cap['url'],note='實際200內文具名現職與2026參選情境；不以搜尋摘要冒稱內文')
review=pd.DataFrame(review);evidence=pd.DataFrame(obs);print(review[['candidate_name','birth_date','sex','education_level_2022']].to_string(index=False))


candidate_name birth_date sex education_level_2022
           吳仁祥 1971-11-07   男                   大學
           莊國輝 1990-10-10   男                   大學
           宋萬富 1955-08-27   男              高中(職)以下
           李家萱 1975-12-30   女                   碩士
           蔡志堅 1985-03-29   男                   大學
           曾茂照 1964-02-18   男                   大學
           黃士駿 1978-04-30   男                高中(職)
           黃宣寓              男                     
           吳新文 1968-07-11   男                   大學
           李麗美 1963-02-10   女                高中(職)


In [3]:
assert council[['candidate_key','party_registration_raw','incumbency_2026']].equals(before[['candidate_key','party_registration_raw','incumbency_2026']])
assert council.birth_date_supported_history.ne('').sum()==703
assert (council.birth_date_supported_history.ne('')|council.birth_date_additional_official.ne('')).sum()==716
assert (council.sex_supported_history.ne('')|council.sex_additional_source_value.ne('')).sum()==717
assert (council.moi_profile_url.eq('')&council.campaign_profile_url.eq('')&council.additional_profile_url.eq('')).sum()==720
assert council.first_time_any_office_supported.eq('0').sum()==721 and council.first_time_this_office_supported.eq('0').sum()==705
for a in plan:
 c=council[council.candidate_key.eq(a['candidate_key'])].iloc[0];assert c.first_time_this_office_supported==''
outputs={'data/processed/candidate_master_current.csv':master,'data/processed/candidate_fields_current.csv':fields,'data/processed/council_background_20261019.csv':council,'data/processed/municipal_roles_review_round9_20261019.csv':review,'data/processed/candidate_evidence_round9_20261019.csv':evidence};checks={}
for p,d in outputs.items():
 assert p not in PINS
 d=(d.map if hasattr(d,'map') else d.applymap)(lambda z:str(z).replace('\r','').replace('\n','\\n'))
 d.to_csv(ROOT/p,index=False,encoding='utf-8-sig');saved=read(p);assert saved.shape==d.shape and saved.columns.tolist()==d.columns.tolist()
 key=['candidate_key','field'] if p.endswith('candidate_fields_current.csv') else [] if 'candidate_evidence_round9' in p else ['candidate_key']
 if key:assert not saved.duplicated(key).any()
 checks[p]={'rows':len(saved),'columns':len(saved.columns),'key':key,'sha256':sha(p)}
summary={'new_government_person_or_role_entries':10,'new_complete_official_birthdays':9,'new_explicit_sex_people':10,'historical_2022_council_people':703,'official_birth_support_people':716,'explicit_sex_support_people':717,'remaining_person_entry_gaps':720,'nonfirst_any':721,'nonfirst_council':705,'pending_identity_unchanged':151,'outputs':checks}
(ROOT/'data/code/municipal_roles_round9_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2));print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "new_government_person_or_role_entries": 10,
  "new_complete_official_birthdays": 9,
  "new_explicit_sex_people": 10,
  "historical_2022_council_people": 703,
  "official_birth_support_people": 716,
  "explicit_sex_support_people": 717,
  "remaining_person_entry_gaps": 720,
  "nonfirst_any": 721,
  "nonfirst_council": 705,
  "pending_identity_unchanged": 151,
  "outputs": {
    "data/processed/candidate_master_current.csv": {
      "rows": 1583,
      "columns": 34,
      "key": [
        "candidate_key"
      ],
      "sha256": "960d538e9af2c15430f4c25f20c523f0e752f2fa16b5492f00b6f9d8c4e65dd4"
    },
    "data/processed/candidate_fields_current.csv": {
      "rows": 31660,
      "columns": 11,
      "key": [
        "candidate_key",
        "field"
      ],
      "sha256": "3d0a35feebd892772286c646ae7b3b0c3bf8e1a0d7f2d0b74c7825f253b9fe46"
    },
    "data/processed/council_background_20261019.csv": {
      "rows": 1502,
      "columns": 50,
      "key": [
        "candidate_key"
 

2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。